<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [3]:
import torch.nn.functional as F

In [23]:
df = pd.read_csv('mnist_train.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([60000, 784]), torch.Size([60000]), torch.float32, torch.int64)

In [47]:
#check kamming_normal initialisation
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100) * 0.01
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10) * 0

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [48]:
print(sum(p.numel() for p in parameters))

79510


In [49]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

In [54]:
#forward pass
batch_size = 50
for i in range(20000): # Reduced iterations for faster debugging
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb = Xtr[ix]
  Yb = Ytr[ix]

  Xb_mean = Xb.mean(0, keepdim=True)
  Xb_std = Xb.std(0, keepdim=True)

  X_mean = 0.998 * X_mean + 0.002 * Xb_mean
  X_std = 0.998 * X_std + 0.002 * Xb_std

  Xb = (Xb - Xb_mean) / (Xb_std + eps)

  hpreact = Xb @ w1 + b1

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Yb)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 15000 else 0.01

  for p in parameters:
    p.data += -lr * p.grad

  if i%100 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

0.010617307387292385
0.018923508003354073
0.012533027678728104
0.04066266119480133
0.004829544108361006
0.05795885995030403
0.00966225191950798
0.0053093405440449715
0.04830613732337952
0.12210769951343536
0.04466451331973076
0.00616403529420495
0.02066817693412304
0.029520833864808083
0.020436914637684822
0.0044459812343120575
0.03336045518517494
0.014991991221904755
0.023439662531018257
0.013051563873887062
0.004808072466403246
0.02335560880601406
0.02551816962659359
0.013219502754509449
0.008688596077263355
0.015407801605761051
0.0062956917099654675
0.011172423139214516
0.02702973037958145
0.022432716563344002
0.03228233382105827
0.020598964765667915
0.007415245287120342
0.012661325745284557
0.05794971436262131
0.02483641915023327
0.04086894169449806
0.01265393104404211
0.1316128671169281
0.009638587944209576
0.03357440605759621
0.018445931375026703
0.019785992801189423
0.004909863229840994
0.04037875309586525
0.029472969472408295
0.060881149023771286
0.02428634651005268
0.007742711

Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [55]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10000, 784])
Test data shape (labels): torch.Size([10000])


In [56]:
X_test = (X_test - X_mean) / (X_std + eps)

In [57]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(hpreact_test)
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Y_test[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 9545 
 Unmatched: 455
